# CO_violation_rates — Raw vs Corrected Violation Rates

Side-by-side bar chart of raw versus corrected violation rates
for 9 forecasters at alpha = 0.01, averaged across 24 assets
(Figure 4).

**Output:** `fig_violation_rates.pdf`/`.png`

In [ ]:
"""
Raw vs corrected violation rates (Figure 4).
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Self-contained plotting style — duplicated across figure Quantlets
# by Q² convention.
C_GRAY = '#666666'
C_TEAL = '#00A651'
C_RED = '#E31E24'
C_BLUE = '#0066CC'
C_PURPLE = '#7B2FBE'

plt.rcParams.update({
    'font.family': 'serif', 'axes.grid': False,
    'savefig.transparent': True, 'savefig.dpi': 300,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 10,
})

MODEL_ORDER = ['Chronos-Small', 'Chronos-Mini', 'TimesFM-2.5',
               'Moirai-2.0', 'Lag-Llama',
               'GJR-GARCH', 'GARCH-N', 'Hist-Sim', 'EWMA']
MODEL_SHORT = ['Chr-S', 'Chr-M', 'TFM', 'Moirai', 'L-Llama',
               'GJR', 'G-N', 'HS', 'EWMA']

SCRIPT_DIR = Path('.').resolve()
BASE = SCRIPT_DIR.parent
DATA = BASE / 'cfp_ijf_data' / 'paper_outputs' / 'tables'
FIG_DIR = BASE / 'figures'
OUT = SCRIPT_DIR

df = pd.read_csv(DATA / 'all_results.csv')
d01 = df[df['alpha'] == 0.01].copy()

summary = d01.groupby('model').agg(
    pi_raw=('pihat_raw', 'mean'),
    pi_cp=('pihat_cp', 'mean'),
).reindex(MODEL_ORDER)

x = np.arange(len(MODEL_ORDER))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(x - w / 2, summary['pi_raw'], w,
       color=C_RED, alpha=0.7, label='Raw')
ax.bar(x + w / 2, summary['pi_cp'], w,
       color=C_TEAL, alpha=0.7, label='Corrected')
ax.axhline(0.01, color=C_GRAY, ls='--', lw=1, label='1% target')

ax.set_title(r'Raw vs Corrected Violation Rates ($\alpha = 0.01$)',
             fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(MODEL_SHORT, fontsize=9)
ax.set_ylabel(r'Mean violation rate $\hat{\pi}$')
ax.set_ylim(0, min(summary['pi_raw'].max() * 1.1, 0.5))

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1),
          ncol=3, fontsize=9, frameon=False)

FIG_DIR.mkdir(exist_ok=True)
for ext in ['pdf', 'png']:
    fig.savefig(OUT / f'fig_violation_rates.{ext}', bbox_inches='tight')
    fig.savefig(FIG_DIR / f'fig_violation_rates.{ext}', bbox_inches='tight')

plt.show()
plt.close(fig)
print('Saved: fig_violation_rates.pdf/.png')